In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df=spark.read.csv("Files/bronze/fraud_detection_credit_card_small.csv",
header=True,
inferSchema=True)

In [ ]:
df.show()

In [ ]:
display(df.limit(5))

In [ ]:
df.printSchema()

In [ ]:
df.count()

In [ ]:
len(df.columns)

In [ ]:
df.filter(df["merch_zipcode"].isNull()).count()

In [ ]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [ ]:
from pyspark.sql.functions import col, sum
null_count= df.select([
sum(col(c).isNull().cast("int")).alias(c)
for c in df.columns
])
display(null_count)


In [ ]:
df.printSchema()

In [ ]:
df =spark.read.csv("Files/bronze/fraud_detection_credit_card_small.csv",
header=True,
inferSchema=True
)


In [ ]:
total_rows=df.count()

In [ ]:
unique_rows = df.drop("Unnamed: 0").dropDuplicates().count()

In [ ]:
duplicate_rows = total_rows - unique_rows

In [ ]:
total_rows = df.count()

unique_rows = df.drop("Unnamed: 0").dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

duplicate_rows

In [ ]:
df.select("trans_num").distinct().count()

In [ ]:
df=spark.read.csv("Files/bronze/fraud_detection_credit_card_small.csv",
header=True,
inferSchema=True)

In [ ]:
total_rows=df.count()

In [ ]:
unique_rows=df.drop("Unnamed:0").drop_duplicates().count()

In [ ]:
total_rows=df.count()
unique_rows=df.drop("Unnamed:0").dropDuplicates().count()
duplicate_rows= total_rows-unique_rows
duplicate_rows


In [ ]:
df=spark.read.csv("Files/bronze/fraud_detection_credit_card_small.csv",
header=True,
inferSchema=True)


In [ ]:
display(df.limit(5))

In [ ]:
df_silver = df.drop("Unnamed: 0")

In [ ]:
len(df_silver.columns)

In [ ]:
from pyspark.sql.functions import col

df_silver = df_silver.withColumn(
    "cc_num",
    col("cc_num").cast("string")
    
)


In [ ]:
df_silver.select("cc_num").printSchema()

In [ ]:
len(df_silver.columns)

In [ ]:
from pyspark.sql.functions import col
df_silver=df_silver.withColumn(
    "zip",
    col("zip").cast("string")

)

In [ ]:
df_silver.select("zip").printSchema()

In [ ]:
from pyspark.sql.functions import col
df_silver=df_silver.withColumn(
    "merch_zipcode",
    col("merch_zipcode").cast("int").cast("string")
)

df_silver.select("merch_zipcode").printSchema()

In [ ]:
df_silver=df_silver.fillna({
    "merch_zipcode":"Unknown"
})

In [ ]:
df_silver.filter(df_silver["merch_zipcode"].isNull()).count()

In [ ]:
from pyspark.sql.functions import sha2, col
df_silver=df_silver.withColumn(
    "customer_id",
    sha2(col("cc_num"),256)
)

In [ ]:
df_silver.select("customer_id").show(5, truncate=False)

In [ ]:
df_silver = df_silver.drop(
    "cc_num",
    "first",
    "last",
    "street",
    "dob"
)

In [ ]:
len(df_silver.columns)

In [ ]:
df_silver.select(
    "category",
    "Merchant_Category"
).show(10, truncate=False)

In [ ]:
df.select(
    "trans_date_trans_time",
    "Transaction_Time"
).show(10, truncate=False)

In [ ]:
df.select(
    "dob",
    "trans_date_trans_time",
    "Customer_Age"
).show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

df.select(
    "dob",
    "trans_date_trans_time",
    "Customer_Age",
    (
        F.year("trans_date_trans_time") - F.year("dob")
        - F.when(
            F.date_format("trans_date_trans_time", "MMdd")
            < F.date_format("dob", "MMdd"),
            1
        ).otherwise(0)
    ).alias("age_at_transaction")
).show(10, truncate=False)

In [ ]:
df.select("trans_num").distinct().count()

In [ ]:
silver_with_dob = df_silver.join(
    df.select("trans_num", "dob"),
    on="trans_num",
    how="left"
)

silver_with_dob.select("trans_num", "dob").show(5, truncate=False)

In [ ]:
silver_with_age = (
    silver_with_dob
    .withColumn(
        "age_at_transaction",
        F.year("trans_date_trans_time") - F.year("dob")
        - F.when(
            F.date_format("trans_date_trans_time", "MMdd")
            < F.date_format("dob", "MMdd"),
            1
        ).otherwise(0)
    )
    .drop("dob", "Customer_Age", "Transaction_Time")
)

silver_with_age.select(
    "trans_date_trans_time",
    "age_at_transaction"
).show(5, truncate=False)

In [ ]:
print("عدد الصفوف:", silver_with_age.count())

print(
    "أعمار ناقصة:",
    silver_with_age.filter(F.col("age_at_transaction").isNull()).count()
)

print(
    "أعمار تحتاج مراجعة:",
    silver_with_age.filter(
        (F.col("age_at_transaction") < 0)
        | (F.col("age_at_transaction") > 120)
    ).count()
)


In [ ]:
df_silver = silver_with_age.drop("Merchant_Category")

print("عدد الصفوف:", df_silver.count())
print("عدد الأعمدة:", len(df_silver.columns))

In [ ]:
df_silver.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("fraud_transactions_silver")

In [ ]:
saved_silver = spark.table("fraud_transactions_silver")
print("الصفوف:", saved_silver.count())
print("الأعمدة:", len(saved_silver.columns))